# Improving 4CH segmentation, round 2: more annotated data, finer grid

Round 1 `recipe_t5` reached 0.900 test fg Dice (3-seed ensemble 0.903) vs 0.880 for the retrained old recipe. Round 2 (configs/improve_round2.yaml) adds `t5_extra` (+15 4CH_VAL cases), `t5_full` (+ the 16 validation cases), `t5_extra_hr` (`t5_extra` on a 1.5x grid) and `t5_hr` (`recipe_t5` on the 1.5x grid; runs without 4CH_VAL).

Needs a **GPU**, **Internet**, the raw MRI dataset and a dataset containing a 4CH_VAL folder with image/ and anno/.

In [ ]:
from pathlib import Path
import subprocess
import sys

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert INPUT.is_dir() and WORK.is_dir(), "Run this notebook on Kaggle."

REPO = WORK / "cardiac-mri-4ch-segmentation"
COMMIT = "b8926456a43d72cfe5201ccca447efd9705066b3"  # commit containing configs/improve_round2.yaml
URL = "https://github.com/sancia-jw/cardiac-mri-4ch-segmentation.git"
assert COMMIT != "SET_AFTER_PUSH", "Pin COMMIT to the pushed commit first."

if not REPO.exists():
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", COMMIT], check=True)

head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == COMMIT, f"Existing checkout is {head}; expected {COMMIT}."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel", "PyYAML", "segmentation-models-pytorch"], check=True)
print("Repository and dependencies ready:", head)

Locate the data. DATA must contain 4CH_TR; EXTRA must contain 4CH_VAL (optional).

In [ ]:
import os

DATA_OVERRIDE = None   # e.g. "/kaggle/input/cmr-multi-cine-4ch/CMR-MULTI/CINE_MULTI"
EXTRA_OVERRIDE = None  # e.g. "/kaggle/input/cmr-multi-cine-4ch-val"

def roots_with(view):
    # os.walk(followlinks=True): Kaggle can mount datasets as symlinks, which Path.rglob skips.
    return sorted({Path(root) for root, dirs, _ in os.walk(INPUT, followlinks=True)
                   if view in dirs and (Path(root) / view / "image").is_dir() and (Path(root) / view / "anno").is_dir()})

found = roots_with("4CH_TR")
DATA = Path(DATA_OVERRIDE) if DATA_OVERRIDE else (found[0] if len(found) == 1 else None)
if DATA is None:
    raise RuntimeError(f"Expected one folder containing 4CH_TR, found {found}. Attach the raw MRI dataset.")
extra = roots_with("4CH_VAL")
EXTRA = Path(EXTRA_OVERRIDE) if EXTRA_OVERRIDE else (extra[0] if extra else None)
OUT = WORK / "improve_round2"
LOGS = WORK / "improve_round2_logs"
CONFIG = REPO / "configs/improve_round2.yaml"
print("DATA:", DATA)
print("EXTRA:", EXTRA or "NOT attached: only t5_hr will run. Attach your 4CH_VAL dataset.")

Check the GPU and inputs (case split, image sizes, cine loops). Nothing is trained.

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a GPU in Kaggle notebook settings."
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
LOGS.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 43, 44]
BUDGET_HOURS = 11.3  # Kaggle stops a session at 12 h; runs that would not finish are not started

def command(*extra_args):
    cmd = [sys.executable, "-u", str(REPO / "scripts/train_improve.py"), "--config", str(CONFIG),
           "--data-root", str(DATA), "--output-root", str(OUT), "--seeds", *map(str, SEEDS)]
    if EXTRA:
        cmd += ["--extra-root", str(EXTRA)]
    return cmd + list(extra_args)

def run_logged(cmd, log_name):
    log_path = LOGS / log_name
    print("Running:", " ".join(cmd))
    with log_path.open("w", encoding="utf-8") as log, subprocess.Popen(
            cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = proc.wait()
    if code:
        raise RuntimeError(f"Command exited with code {code}; see {log_path}")

run_logged(command("--validate-only"), "validation.log")

**Training.** Finished runs are skipped, so re-running continues where a session stopped.

In [ ]:
run_logged(command("--budget-hours", str(BUDGET_HOURS)), "training.log")

Results vs round 1 (legacy seed mean 0.8796; recipe_t5 0.9001, ensemble 0.9034).

In [ ]:
import pandas as pd
from IPython.display import display

LEGACY, T5, T5_ENS = 0.8796, 0.9001, 0.9034
s = pd.read_csv(OUT / "summary.csv", dtype={"seed": str})
names = {"dice_1": "LV", "dice_2": "LV_myo", "dice_3": "RV", "dice_4": "RA", "dice_5": "LA"}
seeds = s[s.seed != "ensemble"]
order = list(dict.fromkeys(s.condition))
table = seeds.groupby("condition").agg(seeds=("seed", "count"), fg=("fg", "mean"), fg_sd=("fg", "std"),
                                       fg_lcc=("fg_lcc", "mean"),
                                       **{v: (k, "mean") for k, v in names.items()}).reindex(order)
table["vs_recipe_t5"] = table["fg"] - T5
table["rel_%_vs_legacy"] = 100 * (table["fg"] - LEGACY) / LEGACY
ens = s[s.seed == "ensemble"].set_index("condition")[["fg", "fg_lcc", *names]].rename(columns=names)
ens["vs_recipe_t5_ens"] = ens["fg"] - T5_ENS
ens["rel_%_vs_legacy"] = 100 * (ens["fg"] - LEGACY) / LEGACY
table.round(4).to_csv(OUT / "seed_table.csv")
ens.round(4).to_csv(OUT / "ensemble_table.csv")
display(table.round(4))
display(ens.round(4))